In [1]:
import pandas as pd

# Creamos un DataFrame base con datos de estudiantes
datos = {
    'id': [101, 102, 103, 104],
    'nombre': ['Ana', 'Juan', 'María', 'Pedro'],
    'nota': [8, 9, 7, 11]  # Le ponemos un 11 a Pedro a propósito
}
df = pd.DataFrame(datos)

display(df)

,id,nombre,nota
0,101,Ana,8
1,102,Juan,9
2,103,María,7
3,104,Pedro,11


### 1. Dos nombres para la misma tabla y `.tolist()`

Cuando en Pandas hacemos una asignación simple como `df_vista = df`, **no estamos creando una tabla nueva**. Estás creando una nueva "etiqueta" que apunta exactamente al mismo espacio en la memoria. Es como si a un estudiante lo llamaran por su nombre y por su apodo; la persona es la misma. Si cambias una nota en la vista, el cambio se refleja en la tabla original.

**¿Y qué es `.tolist()`?**
Es un método que convierte una columna de Pandas en una **lista tradicional de Python** (las que van entre corchetes `[ ]`). Sirve muchísimo si necesitas sacar los datos de la tabla para usarlos en un bucle `for` clásico o en otra herramienta de Python puro.

In [2]:
# Asignación simple (le damos un "apodo" a la misma tabla)
df_vista = df

# Cambiamos la nota del primer estudiante (índice 0)
df_vista.loc[0, 'nota'] = 10

print("DataFrame Original (nota cómo también cambió la primera nota):")
display(df)

# Uso de .tolist() para sacar la columna 'nota' hacia una lista de Python
lista_notas = df['nota'].tolist()
print("\nColumna extraída como lista de Python:", lista_notas)

DataFrame Original (nota cómo también cambió la primera nota):


,id,nombre,nota
0,101,Ana,10
1,102,Juan,9
2,103,María,7
3,104,Pedro,11



Columna extraída como lista de Python: [10, 9, 7, 11]


### 2 y 3. El clon independiente (`.copy()`) y la comparación celda a celda (`==`)

Para evitar el problema anterior, usamos el método `.copy()`. Al hacer esto, ahora sí estamos creando un "clon" independiente. Si modificamos la copia, la tabla original ni se entera.

Lo interesante pasa cuando queremos comparar el original con la copia usando el operador de igualdad (`==`). Pandas no te devuelve un simple `True` o `False` general. En cambio, hace una revisión **celda por celda** y te devuelve una tabla entera mostrándote exactamente qué datos son iguales y cuáles cambiaron.

In [3]:
# Ahora sí creamos un clon independiente
df_copia = df.copy()

# Modificamos SOLO la copia: Cambiamos el 11 por un 10 en la última fila
df_copia.loc[3, 'nota'] = 10

print("DataFrame Original (Pedro sigue teniendo 11):")
display(df)
print("\nDataFrame Copia (Pedro ahora tiene 10):")
display(df_copia)

# Comparamos celda por celda
print("\nComparación lógica (df == df_copia):")
display(df == df_copia)

DataFrame Original (Pedro sigue teniendo 11):


,id,nombre,nota
0,101,Ana,10
1,102,Juan,9
2,103,María,7
3,104,Pedro,11



DataFrame Copia (Pedro ahora tiene 10):


,id,nombre,nota
0,101,Ana,10
1,102,Juan,9
2,103,María,7
3,104,Pedro,10



Comparación lógica (df == df_copia):


,id,nombre,nota
0,True,True,True
1,True,True,True
2,True,True,True
3,True,True,False


### 4. Borrar con red de seguridad: `.drop()` como forma de copia

El método `.drop()` se usa para eliminar filas o columnas. Pero tiene una particularidad genial: **por defecto, no arruina tus datos crudos**. En lugar de modificar tu tabla original directamente, te devuelve una **copia nueva** con esa columna ya eliminada. Es una forma muy segura de trabajar.

In [4]:
# Eliminamos la columna 'nombre'
df_sin_nombres = df.drop(columns=['nombre'])

print("Nuevo DataFrame sin la columna nombres:")
display(df_sin_nombres)

print("\nEl DataFrame original sigue intacto:")
display(df)

Nuevo DataFrame sin la columna nombres:


,id,nota
0,101,10
1,102,9
2,103,7
3,104,11



El DataFrame original sigue intacto:


,id,nombre,nota
0,101,Ana,10
1,102,Juan,9
2,103,María,7
3,104,Pedro,11


### 5. La asignación que conviene evitar (Asignación Encadenada)

Este es el error más común en Pandas. Consiste en usar dos pares de corchetes consecutivos, como esto: `df[df["id"] == 104]["nota"] = 10`.

¿Por qué está mal? Porque para Python estas son **dos operaciones distintas**: 
1. La primera crea una tabla intermedia en la memoria (busca las filas con id 104).
2. La segunda intenta guardarle un valor a esa tabla intermedia que acaba de nacer. 

Pandas se confunde. No sabe si guardaste el dato en tu tabla original o en esa copia temporal que pronto va a desaparecer. Cuando haces esto, Pandas lanza una alerta roja muy famosa llamada `SettingWithCopyWarning` para avisarte que lo que estás haciendo es impredecible (a veces funciona, a veces no).

**La solución correcta:** Usar `.loc[fila, columna]`. Así le damos a Pandas **una sola instrucción directa** a través de la coma. Cero tablas intermedias y cero confusiones.

*Nota: Al ejecutar la siguiente celda de código, verás aparecer el cartel rojo de advertencia a propósito.*

In [5]:
# Creamos un DataFrame de prueba rápido para este ejemplo
df_prueba = pd.DataFrame({
    'id': [101, 102, 103, 104],
    'nota': [8, 9, 7, 11]  # Queremos cambiar este 11 por un 10
})

print("--- FORMA INCORRECTA ---")
# Esto va a generar la alerta roja: SettingWithCopyWarning
df_prueba[df_prueba["id"] == 104]["nota"] = 10

# Mostramos el DataFrame para ver si el cambio se guardó
# (Spoiler: a menudo la nota no cambia y sigue siendo 11)
print("\n¿Cambió la nota realmente con la forma incorrecta?")
display(df_prueba)


print("\n\n--- FORMA CORRECTA ---")
# Usamos .loc separando filas y columnas con una coma
df_prueba.loc[df_prueba["id"] == 104, "nota"] = 10

print("Asignación correcta usando .loc (la nota ahora es 10):")
display(df_prueba)

--- FORMA INCORRECTA ---

¿Cambió la nota realmente con la forma incorrecta?


C:\Users\diodo\AppData\Local\Temp\ipykernel_2700\2084936958.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_prueba[df_prueba["id"] == 104]["nota"] = 10


,id,nota
0,101,8
1,102,9
2,103,7
3,104,11




--- FORMA CORRECTA ---
Asignación correcta usando .loc (la nota ahora es 10):


,id,nota
0,101,8
1,102,9
2,103,7
3,104,10
